<a href="https://colab.research.google.com/github/mishrasatyam9720-debug/ai-inventory-and-retail-price/blob/main/ai_inventory_%26_retail_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import pandas as pd


In [ ]:

np.random.seed(42)

n_products = 20
n_days = 365

dates = pd.date_range(
    start="2025-01-01",
    periods=n_days,
    freq="D"
)


In [ ]:

products = pd.DataFrame({
    "productId": [f"P{i:03d}" for i in range(1, n_products + 1)],
    "name": [f"Product_{i:03d}" for i in range(1, n_products + 1)],
    "category": np.random.choice(
        ["Grocery", "Dairy", "Beverages", "Snacks"], n_products
    ),
    "subcategory": np.random.choice(
        ["Salt", "Milk", "Juice", "Biscuits", "Oil"], n_products
    ),
    "brand": np.random.choice(
        ["Tata", "Amul", "Parle", "Nestle"], n_products
    ),
    "unit": np.random.choice(
        ["1 kg", "500 g", "1 litre", "1 pack"], n_products
    ),
    "purchasePrice": np.random.randint(20, 201, n_products),
    "minimumStock": np.random.randint(20, 101, n_products)
})

products["sellingPrice"] = (
    products["purchasePrice"]
    * np.random.uniform(1.3, 1.8, n_products)
).round(2)

products.head()


,productId,name,category,subcategory,brand,unit,purchasePrice,minimumStock,sellingPrice
0,P001,Product_001,Grocery,Salt,Tata,1 litre,180,46,257.19
1,P002,Product_002,Dairy,Juice,Parle,1 kg,121,40,196.91
2,P003,Product_003,Dairy,Juice,Parle,1 litre,99,50,169.10
3,P004,Product_004,Dairy,Biscuits,Amul,1 litre,123,25,173.23
4,P005,Product_005,Grocery,Milk,Tata,1 litre,85,71,113.46


In [ ]:

df = pd.MultiIndex.from_product(
    [products["productId"], dates],
    names=["productId", "date"]
).to_frame(index=False)

df = df.merge(products, on="productId", how="left")

df.head()



,productId,date,name,category,subcategory,brand,unit,purchasePrice,minimumStock,sellingPrice
0,P001,2025-01-01,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58
1,P001,2025-01-02,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58
2,P001,2025-01-03,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58
3,P001,2025-01-04,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58
4,P001,2025-01-05,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58


In [ ]:

df["discount"] = np.random.choice(
    [0, 5, 10, 15, 20], len(df)
)
df["promotion"] = np.random.choice(
    [0, 1], len(df), p=[0.7, 0.3]
)

weekend = (df["date"].dt.dayofweek >= 5).astype(int)

base_demand = np.random.randint(10, 60, len(df))

df["quantitySold"] = (
    base_demand
    + df["promotion"] * 10
    + df["discount"] * 0.5
    + weekend * 5
    + np.random.normal(0, 5, len(df))
).clip(0).round().astype(int)

df["revenue"] = (
    df["quantitySold"]
    * df["sellingPrice"]
    * (1 - df["discount"] / 100)
).round(2)

df[["date", "productId", "discount",
    "promotion", "quantitySold", "revenue"]].head()


,date,productId,discount,promotion,quantitySold,revenue
0,2025-01-01,P001,10,0,15,3328.83
1,2025-01-02,P001,20,1,79,15583.86
2,2025-01-03,P001,0,0,15,3698.70
3,2025-01-04,P001,10,1,41,9098.80
4,2025-01-05,P001,15,1,62,12994.77


In [ ]:

df["dayOfWeek"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["isWeekend"] = weekend

df[["date", "dayOfWeek", "month", "isWeekend"]].head()



,date,dayOfWeek,month,isWeekend
0,2025-01-01,2,1,0
1,2025-01-02,3,1,0
2,2025-01-03,4,1,0
3,2025-01-04,5,1,1
4,2025-01-05,6,1,1


In [ ]:

df["rolling_mean_7"] = (
    df.groupby("productId")["quantitySold"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

df["rolling_mean_30"] = (
    df.groupby("productId")["quantitySold"]
    .transform(
        lambda x: x.shift(1).rolling(30).mean()
    )
)

df[["productId", "date", "quantitySold",
    "rolling_mean_7", "rolling_mean_30"]]
df.head()



,productId,date,name,category,subcategory,brand,unit,purchasePrice,minimumStock,sellingPrice,...,dayOfWeek,month,isWeekend,rolling_mean_7,rolling_mean_30,expenseType,expenseAmount,expenseDescription,lag_1,lag_7
0,P001,2025-01-01,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,2,1,0,NaN,NaN,Packaging,314,Simulated business expense,NaN,NaN
1,P001,2025-01-02,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,3,1,0,NaN,NaN,Electricity,1362,Simulated business expense,65.0,NaN
2,P001,2025-01-03,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,4,1,0,NaN,NaN,Packaging,1581,Simulated business expense,38.0,NaN
3,P001,2025-01-04,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,5,1,1,NaN,NaN,Transport,532,Simulated business expense,50.0,NaN
4,P001,2025-01-05,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,6,1,1,NaN,NaN,Rent,754,Simulated business expense,38.0,NaN


In [ ]:

df["quantity"] = (
    df["minimumStock"] * 2 - df["quantitySold"]
).clip(lower=0).astype(int)

df["reorderLevel"] = df["minimumStock"]

df[["productId", "date", "minimumStock",
    "quantitySold", "quantity", "reorderLevel"]].head()


,productId,date,minimumStock,quantitySold,quantity,reorderLevel
0,P001,2025-01-01,48,35,61,48
1,P001,2025-01-02,48,54,42,48
2,P001,2025-01-03,48,48,48,48
3,P001,2025-01-04,48,31,65,48
4,P001,2025-01-05,48,61,35,48


In [ ]:

supplier_names = [
    "ABC Distributors",
    "XYZ Traders",
    "Fresh Supply Co."
]

supplier_map = {
    pid: np.random.choice(supplier_names)
    for pid in products["productId"]
}

df["supplierName"] = df["productId"].map(supplier_map)
df["supplierPhone"] = "9876543210"
df["supplierEmail"] = (
    df["supplierName"].str.lower()
    .str.replace(" ", "", regex=False)
    + "@example.com"
)

df["leadTime"] = np.random.randint(1, 8, len(df))

df[["productId", "supplierName",
    "supplierPhone", "supplierEmail", "leadTime"]].head()


,productId,supplierName,supplierPhone,supplierEmail,leadTime
0,P001,ABC Distributors,9876543210,abcdistributors@example.com,3
1,P001,ABC Distributors,9876543210,abcdistributors@example.com,6
2,P001,ABC Distributors,9876543210,abcdistributors@example.com,2
3,P001,ABC Distributors,9876543210,abcdistributors@example.com,7
4,P001,ABC Distributors,9876543210,abcdistributors@example.com,6


In [ ]:

df["purchaseQuantity"] = np.random.randint(10, 51, len(df))

df["totalCost"] = (
    df["purchaseQuantity"] * df["purchasePrice"]
).round(2)

df[["productId", "purchaseQuantity",
    "purchasePrice", "totalCost"]].head()


,productId,purchaseQuantity,purchasePrice,totalCost
0,P001,45,153,6885
1,P001,19,153,2907
2,P001,37,153,5661
3,P001,44,153,6732
4,P001,38,153,5814


In [ ]:

df["expenseType"] = np.random.choice(
    ["Electricity", "Transport", "Rent", "Packaging"],
    len(df)
)

df["expenseAmount"] = np.random.randint(
    100, 2001, len(df)
)

df["expenseDescription"] = "Simulated business expense"

df[["date", "expenseType",
    "expenseAmount", "expenseDescription"]].head()



,date,expenseType,expenseAmount,expenseDescription
0,2025-01-01,Rent,707,Simulated business expense
1,2025-01-02,Packaging,597,Simulated business expense
2,2025-01-03,Electricity,1856,Simulated business expense
3,2025-01-04,Rent,337,Simulated business expense
4,2025-01-05,Rent,1217,Simulated business expense


In [ ]:
df["lag_1"] = df.groupby("productId")["quantitySold"].shift(1)
df["lag_7"] = df.groupby("productId")["quantitySold"].shift(7)

df.head()

,productId,date,name,category,subcategory,brand,unit,purchasePrice,minimumStock,sellingPrice,...,dayOfWeek,month,isWeekend,rolling_mean_7,rolling_mean_30,expenseType,expenseAmount,expenseDescription,lag_1,lag_7
0,P001,2025-01-01,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,2,1,0,NaN,NaN,Packaging,314,Simulated business expense,NaN,NaN
1,P001,2025-01-02,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,3,1,0,NaN,NaN,Electricity,1362,Simulated business expense,65.0,NaN
2,P001,2025-01-03,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,4,1,0,NaN,NaN,Packaging,1581,Simulated business expense,38.0,NaN
3,P001,2025-01-04,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,5,1,1,NaN,NaN,Transport,532,Simulated business expense,50.0,NaN
4,P001,2025-01-05,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,6,1,1,NaN,NaN,Rent,754,Simulated business expense,38.0,NaN


In [ ]:

df.to_csv("retail_sales_dataset.csv", index=False)

print("Dataset saved successfully!")
df.head()
print("Total columns:", len(df.columns))


Dataset saved successfully!
Total columns: 36


In [ ]:
S

,productId,date,name,category,subcategory,brand,unit,purchasePrice,minimumStock,sellingPrice,...,totalCost,expenseType,expenseAmount,expenseDescription,forecastDate,horizon,predictedDemand,modelUsed,lag_1,lag_7
0,P001,2025-01-31,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,7344,Rent,312,Simulated business expense,2025-02-01,7,NaN,Random Forest,34.0,73.0
1,P001,2025-02-01,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,4896,Electricity,1931,Simulated business expense,2025-02-02,7,NaN,Random Forest,19.0,23.0
2,P001,2025-02-02,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,3366,Electricity,485,Simulated business expense,2025-02-03,7,NaN,Random Forest,20.0,46.0
3,P001,2025-02-03,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,7344,Packaging,1535,Simulated business expense,2025-02-04,7,NaN,Random Forest,63.0,38.0
4,P001,2025-02-04,Product_001,Beverages,Oil,Parle,1 kg,153,48,246.58,...,5661,Electricity,1638,Simulated business expense,2025-02-05,7,NaN,Random Forest,45.0,29.0
